# General

For more informations, see the documentation:
- *documentary_strategy*
- *LLM and GenAI*

# Import & Configs

In [1]:
import json
import pandas as pd

In [2]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

In [5]:
%reload_ext autoreload
%autoreload 2

from src.retrieval.retriever import MedicalRetriever

# Questions Set

In [6]:
with open(
    "../data/evaluations/gold_questions.json"
) as f:

    questions = json.load(f)

questions

[{'question': 'What is glioblastoma?'},
 {'question': 'How is glioblastoma prognosis evaluated?'},
 {'question': 'What MRI techniques are used for glioblastoma?'},
 {'question': 'What is peritumoral edema?'},
 {'question': 'What is pseudoprogression?'},
 {'question': 'How is tumor progression detected?'},
 {'question': 'What biomarkers are associated with glioblastoma?'},
 {'question': 'What are the limitations of MRI in glioma diagnosis?'},
 {'question': 'How is overall survival measured?'},
 {'question': 'What treatments are commonly used for glioblastoma?'}]

# Retriever

In [7]:
retriever = MedicalRetriever()

/home/jeremy/Documents/dev/LLM_RAG/Medical_assistant/.ma_env/lib/python3.12/site-packages/torch/cuda/__init__.py:187: UserWarning: CUDA initialization: The NVIDIA driver on your system is too old (found version 12020). Please update your GPU driver by downloading and installing a new version from the URL: http://www.nvidia.com/Download/index.aspx Alternatively, go to: https://pytorch.org to install a PyTorch version that has been compiled with your version of the CUDA driver. (Triggered internally at /pytorch/c10/cuda/CUDAFunctions.cpp:119.)
  return torch._C._cuda_getDeviceCount() > 0


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

# PMID

In [ ]:
results_pmid = []

for item in questions:
    pmid_lst = []
    question = item["question"]
    print(f'{question:-^80}')
    
    output = retriever.retrieve(question, top_k=10)
    print('\n')
    
    for doc in output['metadatas'][0]:
        pmid = doc['pmid']
        pmid_lst.append(pmid)
        print(f'{pmid}')
    print("\n\n")
    
    results_pmid.append({
        "question": question,
        "pmids": pmid_lst
    })

In [ ]:
#results_pmid

In [ ]:
results_pmid_diverse = []

for item in questions:
    pmid_lst = []
    question = item["question"]
    print(f'{question:-^80}')
    
    output = retriever.retrieve_diverse(question, top_k=10)
    print('\n')
    
    for doc in output['metadatas'][0]:
        pmid = doc['pmid']
        pmid_lst.append(pmid)
        print(f'{pmid}')
    print("\n\n")
    
    results_pmid_diverse.append({
        "question": question,
        "pmids": pmid_lst
    })

In [ ]:
#results_pmid_diverse

# Article Retrievers

## Data Upload

In [4]:
def  get_documents_df():
    documents = []
    
    with open("../data/processed/clean_documents.jsonl", "r") as f:
        for line in f:
            documents.append(json.loads(line))

    df = pd.DataFrame(documents)
    return df

In [5]:
df_articles = get_documents_df()
df_articles.head()

,pmid,title,abstract,year,text
0,42200192,Prediction of treatment failure in patients wi...,BACKGROUND: Predicting early treatment failure...,2026,Prediction of treatment failure in patients wi...
1,42200191,Imaging patterns of glioblastoma progression: ...,BACKGROUND: Glioblastoma is the most aggressiv...,2026,Imaging patterns of glioblastoma progression: ...
2,42198443,Antitumor Activity of Cannabinoids and Their I...,Background: Cannabinoids are studied as antica...,2026,Antitumor Activity of Cannabinoids and Their I...
3,42198270,Ultrasound-Enhanced Drug Delivery in Pediatric...,Pediatric brain tumors are highly prevalent an...,2026,Ultrasound-Enhanced Drug Delivery in Pediatric...
4,42195335,Artificial Intelligence-Based MRI Segmentation...,(1) Background: Differentiating between gliobl...,2026,Artificial Intelligence-Based MRI Segmentation...


In [6]:
print(df_articles.columns)
print(df_articles.shape)

Index(['pmid', 'title', 'abstract', 'year', 'text'], dtype='str')
(478, 5)


In [7]:
df_articles.iloc[0]

pmid                                                 42200192
title       Prediction of treatment failure in patients wi...
abstract    BACKGROUND: Predicting early treatment failure...
year                                                     2026
text        Prediction of treatment failure in patients wi...
Name: 0, dtype: str

## PMID dict

In [14]:
def get_pmid_dict(df_articles):
    pmid_to_article = {}

    for _, row in df_articles.iterrows():
    
        pmid_to_article[str(row["pmid"])] = {
            "title": row["title"],
            "text": row["text"],
            "year": row["year"],
        }

    return pmid_to_article

In [15]:
pmid_to_article = get_pmid_dict(df_articles)
print(len(pmid_to_article))

478


In [12]:
print(
    pmid_to_article["42189415"]["text"][:500]
)

Peripheral hematological landscapes as biomarkers for detecting postoperative progression in glioblastoma multiforme: a multivariable risk scoring approach. PURPOSE: Glioblastoma multiforme (GBM) is the most common malignant tumor with poor prognosis despite standard treatment. While various hematological parameters are prognostic for GBM survival, their potential in disease monitoring remains underexplored. Therefore, this study aimed to investigate the value of these parameters in monitoring G


## Articles Retrieval Function

In [9]:
articles = retriever.retrieve_diverse_articles(
    "What is glioblastoma?"
)

for article in articles:
    print(article["pmid"])

Original query: What is glioblastoma?
Processed query: glioblastoma
42135047
42000417
41945366
